In [ ]:
from google.colab import files
uploaded = files.upload()


KeyboardInterrupt: 

In [ ]:
import zipfile
import os

zip_path = "/content/Detection.zip"
extract_path = "/content/dataset"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_path)


In [ ]:
from glob import glob

train_labels = glob('/content/dataset/GTSDB_Train_and_Test/Train/labels/*.txt')
test_labels = glob('/content/dataset/GTSDB_Train_and_Test/Test/labels/*.txt')

label_files = train_labels + test_labels
class_ids = set()

for file in label_files:
    with open(file, 'r') as f:
        for line in f:
            if line.strip():
                class_ids.add(int(line.split()[0]))

class_ids = sorted(list(class_ids))
print("Number of classes:", len(class_ids))
print("Class IDs found:", class_ids)
class_names = [f'class{i}' for i in class_ids]
print("Example class names:", class_names)


In [ ]:
yaml_path = "/content/traffic.yaml"

with open(yaml_path, "w") as f:
    f.write("train: /content/dataset/GTSDB_Train_and_Test/Train/images\n")
    f.write("val: /content/dataset/GTSDB_Train_and_Test/Test/images\n")
    f.write(f"nc: {len(class_names)}\n")
    f.write("names: " + str(class_names) + "\n")

print(" YOLO config created at:", yaml_path)


In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -r requirements.txt


In [ ]:
!python train.py \
  --img 416 \
  --batch 16 \
  --epochs 30 \
  --data /content/traffic.yaml \
  --weights yolov5s.pt \
  --name gtsdb_yolo


In [ ]:
!rm -rf /content/yolov5/runs/train/gtsdb_yolo


In [ ]:
%env WANDB_MODE=disabled
!python train.py \
  --img 416 \
  --batch 16 \
  --epochs 3 \
  --data /content/traffic.yaml \
  --weights yolov5s.pt \
  --name gtsdb_yolo

In [ ]:
!ls /content/yolov5/runs/train/gtsdb_yolo/weights/


In [ ]:
def remap_class_100_to_43(label_folder):
    for file in os.listdir(label_folder):
        full_path = os.path.join(label_folder, file)
        with open(full_path, 'r') as f:
            lines = f.readlines()
        new_lines = []
        for line in lines:
            parts = line.strip().split()
            if int(parts[0]) == 100:
                parts[0] = '43'
            new_lines.append(' '.join(parts) + '\n')
        with open(full_path, 'w') as f:
            f.writelines(new_lines)

remap_class_100_to_43('/content/dataset/GTSDB_Train_and_Test/Train/labels')
remap_class_100_to_43('/content/dataset/GTSDB_Train_and_Test/Test/labels')


In [ ]:
names = [f'class_{i}' for i in range(43)] + ['class_100']
with open('/content/traffic.yaml', 'w') as f:
    f.write(f'''
train: /content/dataset/GTSDB_Train_and_Test/Train/images
val: /content/dataset/GTSDB_Train_and_Test/Test/images
nc: 44
names: {names}
''')


In [ ]:
%cd /content/yolov5
%env WANDB_MODE=disabled

!python train.py --img 416 --batch 16 --epochs 10 \
    --data /content/traffic.yaml \
    --weights yolov5s.pt \
    --name gtsdb_yolo_cleaned


In [ ]:
import torch
from PIL import Image
from IPython.display import display

model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/yolov5/runs/train/gtsdb_yolo_cleaned/weights/best.pt')

image_path = '/content/dataset/GTSDB_Train_and_Test/Test/images/00603.jpg'

model.conf = 0.01

results = model(image_path)

results.print()
results.show()

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/yolov5/runs/train


In [ ]:
from IPython.display import Image, display
display(Image(filename='/content/yolov5/runs/train/gtsdb_yolo_cleaned/results.png'))


In [ ]:
!cat /content/yolov5/runs/train/gtsdb_yolo_cleaned/results.txt


In [ ]:
from google.colab import files
files.download('/content/yolov5/runs/train/gtsdb_yolo_cleaned/weights/best.pt')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
!cp /content/yolov5/runs/train/gtsdb_yolo_cleaned/weights/best.pt /content/drive/MyDrive/


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import zipfile

with zipfile.ZipFile('Recognition.zip', 'r') as zip_ref:
    zip_ref.extractall('/content/recognition_data')


In [ ]:
extract_path = "/content/recognition_data"
train_dir = os.path.join(extract_path, "Train")
test_dir = os.path.join(extract_path, "Test")

print("Train path exists:", os.path.exists(train_dir))
print("Test path exists:", os.path.exists(test_dir))


In [ ]:
!ls /content/recognition_data


In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

data_dir = "/content/recognition_data"
train_csv = os.path.join(data_dir, "Train.csv")
test_csv = os.path.join(data_dir, "Test.csv")
train_dir = os.path.join(data_dir, "Train")
test_dir = os.path.join(data_dir, "Test")

train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

print("Train samples:", len(train_df))
print("Test samples:", len(test_df))


In [ ]:
class TrafficSignDataset(Dataset):
    def __init__(self, dataframe, image_dir, transform=None):
        self.dataframe = dataframe
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        row = self.dataframe.iloc[idx]
        img_path = os.path.join(self.image_dir, str(row['ClassId']), row['Path'].split('/')[-1])
        image = Image.open(img_path).convert('RGB')
        label = row['ClassId']
        if self.transform:
            image = self.transform(image)
        return image, label


In [ ]:

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_df_split, val_df = train_test_split(train_df, test_size=0.2, stratify=train_df['ClassId'])

train_dataset = TrafficSignDataset(train_df_split, train_dir, transform=transform)
val_dataset = TrafficSignDataset(val_df, train_dir, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)


In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * 14 * 14, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 14 * 14)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = SimpleCNN(num_classes=43).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

epochs = 10
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")


In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        outputs = model(images)
        _, preds = torch.max(outputs, 1)
        y_true.extend(labels.numpy())
        y_pred.extend(preds.cpu().numpy())

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(12, 10))
sns.heatmap(cm, annot=False, fmt="d")
plt.title("Confusion Matrix")
plt.show()

print(classification_report(y_true, y_pred, digits=3))


In [ ]:
torch.save(model.state_dict(), "cnn_traffic_sign.pth")


In [ ]:
from PIL import Image
import torchvision.transforms as transforms
class_labels = {
    0: "Speed limit (20km/h)",
    1: "Speed limit (30km/h)",
    2: "Speed limit (50km/h)",
    3: "Speed limit (60km/h)",
    4: "Speed limit (70km/h)",
    5: "Speed limit (80km/h)",
    6: "End of speed limit (80km/h)",
    7: "Speed limit (100km/h)",
    8: "Speed limit (120km/h)",
    9: "No passing",
    10: "No passing for vehicles over 3.5 metric tons",
    11: "Right-of-way at the next intersection",
    12: "Priority road",
    13: "Yield",
    14: "Stop",
    15: "No vehicles",
    16: "Vehicles over 3.5 metric tons prohibited",
    17: "No entry",
    18: "General caution",
    19: "Dangerous curve to the left",
    20: "Dangerous curve to the right",
    21: "Double curve",
    22: "Bumpy road",
    23: "Slippery road",
    24: "Road narrows on the right",
    25: "Road work",
    26: "Traffic signals",
    27: "Pedestrians",
    28: "Children crossing",
    29: "Bicycles crossing",
    30: "Beware of ice/snow",
    31: "Wild animals crossing",
    32: "End of all speed and passing limits",
    33: "Turn right ahead",
    34: "Turn left ahead",
    35: "Ahead only",
    36: "Go straight or right",
    37: "Go straight or left",
    38: "Keep right",
    39: "Keep left",
    40: "Roundabout mandatory",
    41: "End of no passing",
    42: "End of no passing by vehicles over 3.5 metric tons"
}

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

test_image_path = "/content/recognition_data/Test/00002.png"

image = Image.open(test_image_path).convert('RGB')
input_tensor = transform(image).unsqueeze(0)

model.eval()
with torch.no_grad():
    output = model(input_tensor)
    _, predicted_class = torch.max(output, 1)

pred_id = predicted_class.item()
print(f"Predicted Class Name: {class_labels[pred_id]}")


In [ ]:
model = SimpleCNN(num_classes=43)
model.load_state_dict(torch.load("cnn_traffic_sign.pth"))
model.eval()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

torch.save(model.state_dict(), "/content/drive/MyDrive/cnn_traffic_sign.pth")


In [ ]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import cv2
import os
import numpy as np
from matplotlib import pyplot as plt
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * 14 * 14, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 14 * 14)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
cnn_model = SimpleCNN(num_classes=43)
cnn_model.load_state_dict(torch.load("cnn_traffic_sign.pth", map_location=torch.device('cpu')))
cnn_model.eval()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='best.pt', source='local')



In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import torch
yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/drive/MyDrive/best.pt')


In [ ]:
class_labels = [
    "Speed limit (20km/h)", "Speed limit (30km/h)", "Speed limit (50km/h)",
    "Speed limit (60km/h)", "Speed limit (70km/h)", "Speed limit (80km/h)",
    "End of speed limit (80km/h)", "Speed limit (100km/h)", "Speed limit (120km/h)",
    "No passing", "No passing for vehicles over 3.5 metric tons", "Right-of-way at the next intersection",
    "Priority road", "Yield", "Stop", "No vehicles", "Vehicles over 3.5 metric tons prohibited",
    "No entry", "General caution", "Dangerous curve to the left", "Dangerous curve to the right",
    "Double curve", "Bumpy road", "Slippery road", "Road narrows on the right", "Road work",
    "Traffic signals", "Pedestrians", "Children crossing", "Bicycles crossing", "Beware of ice/snow",
    "Wild animals crossing", "End of all speed and passing limits", "Turn right ahead", "Turn left ahead",
    "Ahead only", "Go straight or right", "Go straight or left", "Keep right", "Keep left",
    "Roundabout mandatory", "End of no passing", "End of no passing by vehicles over 3.5 metric tons"
]
def detect_and_classify(image_path):
    results = yolo_model(image_path)
    results.render()
    detections = results.pandas().xyxy[0]

    image = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    for idx, row in detections.iterrows():
        xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])

        cropped = img_rgb[ymin:ymax, xmin:xmax]
        if cropped.size == 0:
            continue

        pil_img = Image.fromarray(cropped)

        input_tensor = transform(pil_img).unsqueeze(0)

        with torch.no_grad():
            output = cnn_model(input_tensor)
            _, pred = torch.max(output, 1)
            class_id = pred.item()
            label = class_labels[class_id]

        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(image, label, (xmin, ymin - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (36, 255, 12), 2)

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Detected and Classified Traffic Signs")
    plt.show()


In [ ]:
from google.colab import files
uploaded = files.upload()


In [ ]:
detect_and_classify("00002.jpg")
results = yolo_model("00002.jpg")
print(results.pandas().xyxy[0])



In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -r requirements.txt

In [ ]:
import torch
yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/drive/MyDrive/best.pt')

In [ ]:
import torch
import torchvision.transforms as transforms
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
import cv2
import os
import numpy as np
from matplotlib import pyplot as plt
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * 14 * 14, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 14 * 14)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [ ]:
cnn_model = SimpleCNN(num_classes=43)
cnn_model.load_state_dict(torch.load("cnn_traffic_sign.pth", map_location=torch.device('cpu')))
cnn_model.eval()
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
cnn_model.load_state_dict(torch.load('/content/drive/MyDrive/cnn_traffic_sign.pth', map_location=torch.device('cpu')))


In [ ]:
cnn_model.eval()
transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

In [ ]:
class_labels = [
    "Speed limit (20km/h)", "Speed limit (30km/h)", "Speed limit (50km/h)",
    "Speed limit (60km/h)", "Speed limit (70km/h)", "Speed limit (80km/h)",
    "End of speed limit (80km/h)", "Speed limit (100km/h)", "Speed limit (120km/h)",
    "No passing", "No passing for vehicles over 3.5 metric tons", "Right-of-way at the next intersection",
    "Priority road", "Yield", "Stop", "No vehicles", "Vehicles over 3.5 metric tons prohibited",
    "No entry", "General caution", "Dangerous curve to the left", "Dangerous curve to the right",
    "Double curve", "Bumpy road", "Slippery road", "Road narrows on the right", "Road work",
    "Traffic signals", "Pedestrians", "Children crossing", "Bicycles crossing", "Beware of ice/snow",
    "Wild animals crossing", "End of all speed and passing limits", "Turn right ahead", "Turn left ahead",
    "Ahead only", "Go straight or right", "Go straight or left", "Keep right", "Keep left",
    "Roundabout mandatory", "End of no passing", "End of no passing by vehicles over 3.5 metric tons"
]
def detect_and_classify(image_path):
    results = yolo_model(image_path)
    results.render()
    detections = results.pandas().xyxy[0]

    image = cv2.imread(image_path)
    img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    for idx, row in detections.iterrows():
        class_id_yolo = int(row['class'])
        if class_id_yolo >= 43:
            continue
        xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
        cropped = img_rgb[ymin:ymax, xmin:xmax]
        if cropped.size == 0:
            continue

        pil_img = Image.fromarray(cropped)

        input_tensor = transform(pil_img).unsqueeze(0)

        with torch.no_grad():
            output = cnn_model(input_tensor)
            _, pred = torch.max(output, 1)
            class_id = pred.item()
            label = class_labels[class_id]

        cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
        cv2.putText(image, label, (xmin, ymin - 10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (36, 255, 12), 2)

    plt.figure(figsize=(10, 10))
    plt.imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    plt.title("Detected and Classified Traffic Signs")
    plt.show()

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
detect_and_classify("00002.jpg")
results = yolo_model("00002.jpg")
print(results.pandas().xyxy[0])

In [ ]:
image_path = "00002.jpg"
detect_and_classify(image_path)


In [ ]:
results = yolo_model("00002.jpg")
print(results.pandas().xyxy[0])


In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
import cv2
resized = cv2.resize(cv2.imread("00002.jpg"), (640, 640))
cv2.imwrite("resized_00002.jpg", resized)
results = yolo_model("resized_00002.jpg")
df = results.pandas().xyxy[0]
print(df[df['confidence'] > 0.05])
results.show()


In [ ]:
!wget -q -c https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-stable-linux-amd64.zip
!unzip -o ngrok-stable-linux-amd64.zip
!mv ngrok /usr/local/bin


In [ ]:
!ngrok authtoken 2zGed58L98jIHFAYvB0sjMNIMpP_TscyxKdjxYSDauFT8RPu

In [ ]:
!pip install flask-ngrok
!ngrok authtoken 2zGed58L98jIHFAYvB0sjMNIMpP_TscyxKdjxYSDauFT8RPu


Authtoken saved to configuration file: /root/.config/ngrok/ngrok.yml


In [ ]:
%%writefile cnn_model.py
import torch
import torch.nn as nn
import torch.nn.functional as F

class SimpleCNN(nn.Module):
    def __init__(self, num_classes=43):
        super(SimpleCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3)
        self.fc1 = nn.Linear(64 * 14 * 14, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64 * 14 * 14)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x


Writing cnn_model.py


In [ ]:
%%writefile app.py
import os
import cv2
import torch
import torchvision.transforms as transforms
from PIL import Image
from flask import Flask, render_template, request
from flask_ngrok import run_with_ngrok
from cnn_model import SimpleCNN

yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/drive/MyDrive/best.pt')
yolo_model.eval()

cnn_model = SimpleCNN(num_classes=43)
cnn_model.load_state_dict(torch.load('/content/drive/MyDrive/cnn_traffic_sign.pth', map_location=torch.device('cpu')))
cnn_model.eval()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])


class_labels = [
    "Speed limit (20km/h)", "Speed limit (30km/h)", "Speed limit (50km/h)",
    "Speed limit (60km/h)", "Speed limit (70km/h)", "Speed limit (80km/h)",
    "End of speed limit (80km/h)", "Speed limit (100km/h)", "Speed limit (120km/h)",
    "No passing", "No passing for vehicles over 3.5 metric tons", "Right-of-way at the next intersection",
    "Priority road", "Yield", "Stop", "No vehicles", "Vehicles over 3.5 metric tons prohibited",
    "No entry", "General caution", "Dangerous curve to the left", "Dangerous curve to the right",
    "Double curve", "Bumpy road", "Slippery road", "Road narrows on the right", "Road work",
    "Traffic signals", "Pedestrians", "Children crossing", "Bicycles crossing", "Beware of ice/snow",
    "Wild animals crossing", "End of all speed and passing limits", "Turn right ahead", "Turn left ahead",
    "Ahead only", "Go straight or right", "Go straight or left", "Keep right", "Keep left",
    "Roundabout mandatory", "End of no passing", "End of no passing by vehicles over 3.5 metric tons"
]

app = Flask(__name__)
run_with_ngrok(app)

UPLOAD_FOLDER = 'static/uploads'
os.makedirs(UPLOAD_FOLDER, exist_ok=True)
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

@app.route('/', methods=['GET', 'POST'])
def index():
    image_path = None
    if request.method == 'POST':
        file = request.files['file']
        filename = file.filename
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)

        results = yolo_model(filepath)
        detections = results.pandas().xyxy[0]
        image = cv2.imread(filepath)
        img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        for idx, row in detections.iterrows():
            class_id_yolo = int(row['class'])
            if class_id_yolo >= 43:
                continue

            xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
            cropped = img_rgb[ymin:ymax, xmin:xmax]
            if cropped.size == 0:
                continue

            pil_img = Image.fromarray(cropped)
            input_tensor = transform(pil_img).unsqueeze(0)
            with torch.no_grad():
                output = cnn_model(input_tensor)
                _, pred = torch.max(output, 1)
                label = class_labels[pred.item()]

            cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
            cv2.putText(image, label, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (36, 255, 12), 2)

        result_path = os.path.join(app.config['UPLOAD_FOLDER'], 'result.jpg')
        cv2.imwrite(result_path, image)
        image_path = '/' + result_path

    return render_template('index.html', image_path=image_path)

if __name__ == '__main__':
    app.run()

Writing app.py


In [ ]:
import os
os.makedirs('templates', exist_ok=True)

with open("templates/index.html", "w") as f:
    f.write('''
<!DOCTYPE html>
<html>
<head>
    <title>Traffic Sign Detector</title>
</head>
<body>
    <h1>Traffic Sign Detection & Classification</h1>
    <form method="POST" enctype="multipart/form-data">
        <input type="file" name="file">
        <input type="submit" value="Upload and Detect">
    </form>

    {% if image_path %}
        <h2>Detected Output:</h2>
        <img src="{{ image_path }}" width="500">
    {% endif %}
</body>
</html>
''')


In [ ]:
!python app.py


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2025-7-1 Python-3.11.13 torch-2.6.0+cu124 CPU

Fusing layers... 
Model summary: 157 layers, 7128793 parameters, 0 gradients, 16.1 GFLOPs
Adding AutoShape... 
 * Serving Flask app 'app'
 * Debug mode: off
 * Running on http://127.0.0.1:5000
Press CTRL+C to quit
Usage of ngrok requires a verified account and authtoken.

Sign up for an account: https://dashboard.ngrok.com/signup
Install your authtoken: https://dashboard.ngrok.com/get-started/your-authtoken

ERR_NGROK_4018

Exception in thread Thread-1:
Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/urllib3/connection.py", line 198, in _new_conn
    sock = connection.create_connection(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/urllib3/util/connection.py", line 85, in create_connection
    raise err
  File "/usr/local/lib/python3.11/dist-packages/urllib3/util/connection.py", line 7

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!mv cloudflared-linux-amd64 cloudflared
!chmod +x cloudflared


In [ ]:
import os

os.makedirs("static/uploads", exist_ok=True)
os.makedirs("templates", exist_ok=True)


In [ ]:
with open("templates/index.html", "w") as f:
    f.write("""<!DOCTYPE html>
<html>
<head>
    <title>Traffic Sign Detection</title>
</head>
<body>
    <h1>Upload a Traffic Sign Image</h1>
    <form method="POST" enctype="multipart/form-data">
        <input type="file" name="file" accept="image/*">
        <input type="submit" value="Upload">
    </form>

    {% if image_path %}
    <h2>Detected and Classified Image:</h2>
    <img src="{{ image_path }}" width="500">
    {% endif %}
</body>
</html>
""")


In [ ]:
!git clone https://github.com/ultralytics/yolov5
%cd yolov5
%pip install -r requirements.txt


Cloning into 'yolov5'...
remote: Enumerating objects: 17496, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (5/5), done.
remote: Total 17496 (delta 2), reused 0 (delta 0), pack-reused 17491 (from 3)
Receiving objects: 100% (17496/17496), 16.54 MiB | 17.48 MiB/s, done.
Resolving deltas: 100% (11990/11990), done.
/content/yolov5
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 49.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 28.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 26.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 889.9 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.4 MB/s eta 0:0

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import torch
import sys
sys.path.append('/content/yolov5')

yolo_model = torch.hub.load('/content/yolov5', 'custom',
                            path='/content/drive/MyDrive/Endterm Assignment/yolo.pt',
                            source='local')



YOLOv5 🚀 2025-7-2 Python-3.11.13 torch-2.6.0+cu124 CPU

Fusing layers... 
Model summary: 157 layers, 7128793 parameters, 0 gradients, 16.1 GFLOPs
Adding AutoShape... 


In [ ]:
from flask import Flask, request, render_template
import os
import cv2
from PIL import Image
import torch
import torchvision.transforms as transforms
import numpy as np
from cnn_model import SimpleCNN
from pathlib import Path

yolo_model = torch.hub.load('ultralytics/yolov5', 'custom', path='/content/drive/MyDrive/Endterm Assignment/yolo.pt')

cnn_model = SimpleCNN()
cnn_model.load_state_dict(torch.load('/content/drive/MyDrive/Endterm Assignment/cnn_traffic_sign.pth', map_location='cpu'))
cnn_model.eval()

transform = transforms.Compose([
    transforms.Resize((64, 64)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

class_labels = [ "Speed limit (20km/h)", "Speed limit (30km/h)", "Speed limit (50km/h)", ... ]

app = Flask(__name__)
UPLOAD_FOLDER = 'static/uploads'
app.config['UPLOAD_FOLDER'] = UPLOAD_FOLDER

@app.route('/', methods=['GET', 'POST'])
def index():
    image_path = None
    if request.method == 'POST':
        file = request.files['file']
        filename = file.filename
        filepath = os.path.join(app.config['UPLOAD_FOLDER'], filename)
        file.save(filepath)

        results = yolo_model(filepath)
        detections = results.pandas().xyxy[0]
        image = cv2.imread(filepath)
        img_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        for idx, row in detections.iterrows():
            class_id_yolo = int(row['class'])
            if class_id_yolo >= 43: continue

            xmin, ymin, xmax, ymax = int(row['xmin']), int(row['ymin']), int(row['xmax']), int(row['ymax'])
            cropped = img_rgb[ymin:ymax, xmin:xmax]
            if cropped.size == 0: continue

            pil_img = Image.fromarray(cropped)
            input_tensor = transform(pil_img).unsqueeze(0)
            with torch.no_grad():
                output = cnn_model(input_tensor)
                _, pred = torch.max(output, 1)
                label = class_labels[pred.item()]

            cv2.rectangle(image, (xmin, ymin), (xmax, ymax), (0, 255, 0), 2)
            cv2.putText(image, label, (xmin, ymin - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)

        result_path = os.path.join(app.config['UPLOAD_FOLDER'], 'result.jpg')
        cv2.imwrite(result_path, image)
        image_path = '/' + result_path

    return render_template('index.html', image_path=image_path)

if __name__ == '__main__':
    app.run()


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master
YOLOv5 🚀 2025-7-2 Python-3.11.13 torch-2.6.0+cu124 CPU

Fusing layers... 
Model summary: 157 layers, 7128793 parameters, 0 gradients, 16.1 GFLOPs
Adding AutoShape... 


 * Serving Flask app '__main__'
 * Debug mode: off


Address already in use
Port 5000 is in use by another program. Either identify and stop that program, or start the server with a different port.
ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/werkzeug/serving.py", line 759, in __init__
    self.server_bind()
  File "/usr/lib/python3.11/http/server.py", line 136, in server_bind
    socketserver.TCPServer.server_bind(self)
  File "/usr/lib/python3.11/socketserver.py", line 472, in server_bind
    self.socket.bind(self.server_address)
OSError: [Errno 98] Address already in use

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/tmp/ipython-input-24-1260042212.py", line 73, in <cell line: 0>
    app.run()
  File "/usr/local/lib/python3.11/dist-packages/flask/app.py", line 662, in run
    run_simple(t.cast(str, host), port, self, **options)
  File "/usr/local/lib/python3.11/dist-packages/werkzeug/serving.py", line 1093, in run_si

TypeError: object of type 'NoneType' has no len()